In [1]:
import sys
import asyncio
from dotenv import load_dotenv

load_dotenv()

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":  # 2. 修复 Windows 下的异步子进程问题

    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:  # 3. 修复 Jupyter Notebook 里的流错误

        sys.stderr = sys.__stderr__

# MCP (Model Context Protocol)

MCP (Model Context Protocol) 是由 Anthropic 推出的开放标准，用于便捷的将AI应用连接外部系统。

在没有MCP的时候，必须手动定义工具，用工具实现文件操作、web搜索、查询航班、查询天气等功能，从而让AI连接外部系统。
但全世界有各种不同的服务，不同服务接口不同，定义Tool非常麻烦。

MCP就像是AI世界的USB接口协议：
- 所有外部服务提供者都可以遵循MCP协议提供Tool
- AI应用基于MCP协议对接任意遵循MCP的外部服务，无需自己定义Tool，实现了工具的快速插拔


## 1.MCP核心概念

| 概念             | 说明                                       |
|----------------|------------------------------------------|
| **MCP Server** | 提供MCP服务的应用，可以是远程服务，也可以是本地服务              |
| **MCP Client** | 连接到MCP服务器，读取MCP信息（特别是Tool信息），供Host使用     |
| **MCP Host**   | 协调和管理多个MCP Client的AI应用，比如LangChain的Agent |

例如，一个AI应用，包含一个操作本地文件的MCP、一个访问数据库的MCP、一个访问外部服务的MCP

![image.png](attachment:60688eea-6be6-4154-b7ad-ee4001db40b1.png)


详细的MCP说明可以参考Anthropic公司的MCP官方文档：
https://modelcontextprotocol.io/docs/getting-started/intro

LangChain本身实现了对MCP的支持，本节我们就来学习如何在LangChain中使用MCP


## 2. 连接外部MCP服务

很多提供云服务的公司都提供了MCP服务，例如：
- Amap Maps : 高德地图提供的MCP
- Filesystem : 可以操作文件系统的MCP
- Time : 查询当前时间的MCP服务
- Kiwi : 查询航班、预定航班的MCP服务
- ...

大家可以在mcp.so搜索各种MCP服务：https://mcp.so/

找好自己想要使用的MCP服务后，就可以用LangChain来对接了。

首先，我们需要安装LangChain的MCP依赖库:
```
uv add langchain-mcp-adapters
```

接下来，就可以利用MCP提供的工具来创建Agent了。


### 2.1 Time MCP服务

Time是一个提供时间和时区转换功能的模型上下文协议服务器。此服务器使语言模型能够获取当前时间信息，并使用 IANA 时区名称执行时区转换，同时具备自动检测系统时区的功能。

需要注意的是，Time MCP是基于stdio的通信方式，也就是把这个MCP服务的脚本下载到本地，然后作为一个子进程运行。此时Agent与MCP服务之间的通信就是进程通信，没有网络延迟。

MCP脚本下载的方式取决于Command，常见的有两种：
- npx : 基于node.js的包管理工具
- uvx : 基于uv(python的uv工具)的包管理工具

因此，你的本地环境必须支持npx、uvx命令

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 1.获取工具
# 1.1定义MCP Client，连接time MCP服务
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)

# 1.2 用client拉取tool，注意MultiServerMCPClient是异步的，需要await
tools = await client.get_tools()


In [3]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print('------------------------')

    # 看输出的样子我们已经成功拿到工具了，接下来只要把这个工具注册到agent就可以了

get_current_time
Get current time in a specific timezone
------------------------
convert_time
Convert time between timezones
------------------------
get_time_ago
Get the time a given number of days, hours, minutes and seconds ago in a specific timezone
------------------------


In [5]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

# 2.初始化模型
model = init_chat_model(
    model="deepseek-flash",
    extra_body={"thinging": {"type": "disabled"}}
)

# 3.创建agent
agent = create_agent(model=model, tools=tools, )


In [6]:
from langchain_core.messages import HumanMessage

# MCP是异步的，所以要使用MCP也得是异步的，调异步函数，所以是ainvoke
response = await agent.ainvoke({"messages": [HumanMessage("现在几点了")]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

现在几点了
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_CY9DS5LELw9lesvgEgUe3790)
 Call ID: call_00_CY9DS5LELw9lesvgEgUe3790
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-05T17:27:43+08:00",\n  "day_of_week": "Thursday",\n  "is_dst": false\n}', 'id': 'lc_f48bb54e-8e78-4962-8d9b-15fbd9426594'}]
================================== Ai Message ==================================

现在是 **2026年11月5日（星期四）17:27:43**，北京时间（UTC+8）。

如果你在其他时区，告诉我，我可以帮你换算。


### 2.2 Kiwi MCP服务

Kiwi Travel MCP 将 Kiwi.com 的航班搜索功能直接引入到 AI 对话中。支持单程/往返、灵活日期、多名乘客以及所有舱位等级。

注意：Kiwi服务是基于http的通信方式，也就是说MCP client通过发送http请求与MCP server交互。存在网络延迟。

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 连接Kiwi mcp服务器
client = MultiServerMCPClient(
    {
        "kiwi-com-flight-search": {
            "transport": "http",  # 注意在langchain中是必须添加这个transport模式的
            "url": "https://mcp.kiwi.com"
        },
        "time": {
            "transport": "stdio",
            "command": "uvx",
            "args": [
                "mcp-server-time",
                "--local-timezone=Asia/Shanghai"
            ]
        }
    }
)
# 注意MultiServerMCPClient是异步的，结果都是协程对象，需要await
tools = await client.get_tools()

In [9]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print('---------------------')

search-flight
# Search for flights

Searches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.

## Result shape

Returns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:
- `price` (number) and `priceFormatted` (e.g. "123 EUR")
- `totalDurationSeconds`
- `bookingUrl` — the link to book the flight
- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)
- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)
- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`, `

In [10]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="你是一名旅行Agent。帮助用户找到最好的航班。不要追问用户问题。如果用户使用中文，则使用zh-cn作为区域设置值。"
)

In [12]:
from langchain_core.messages import HumanMessage

response = await agent.ainvoke(
    {"messages": [HumanMessage("查一下下周天从哈尔滨飞往深圳的航班")]}
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

查一下下周天从哈尔滨飞往深圳的航班
================================== Ai Message ==================================

我先确认一下当前日期，以便算出“下周天”的准确日期。
Tool Calls:
  get_current_time (call_00_TyTtz9gFZ12mM1QjmWPc5398)
 Call ID: call_00_TyTtz9gFZ12mM1QjmWPc5398
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-05T17:44:01+08:00",\n  "day_of_week": "Thursday",\n  "is_dst": false\n}', 'id': 'lc_801afac4-a494-4fd1-8231-a333699b0806'}]
================================== Ai Message ==================================

当前是 2026年11月5日（周四），“下周天”即 **2026年11月15日（周日）**。我来搜索哈尔滨 → 深圳的航班。
Tool Calls:
  search-flight (call_00_FTVVzpJce4OwZYSxOikS7334)
 Call ID: call_00_FTVVzpJce4OwZYSxOikS7334
  Args:
    flyFrom: 哈尔滨
    flyTo: 深圳
    departureDate: 15/11/2026
    locale: 

### 2.3 总结

MCP服务有两种通信方式：
- stdio : 将MCP server作为子进程，通过stdin/stdout通信
- http: 通过http方式与MCP server通信，适合于第三方远程服务，支持SSE


### 3.1 创建简单的MCP服务器

自定义MCP最简单的方式就是使用FastMCP了。
首先，需要安装依赖：
```
uv add fastmcp
```
接着，只需要定义几个方法，然后利用FastMCP提供的装饰器即可：
- @mcp.tool : 作为MCP中的工具
- @mcp.resources : 返回MCP需要的resources
- @mcp.prompt : 返回MCP预定义的Prompt

通常我们只需要定义带有tool的MCP Server就可以了。


In [15]:
from fastmcp import FastMCP

# 初始化mcp
mcp = FastMCP("math")


# mcp tools
@mcp.tool
def add(a: float, b: float) -> float:
    """add two numbers"""
    return a + b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """Multiply two numbers"""
    return a * b


@mcp.tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5


# 启动mcp服务，通信方式设为stdio
if __name__ == '__main__':
    mcp.run(transport="stdio")


ImportError: FastMCP server support is not installed. Install `fastmcp` or `fastmcp-slim[server]`.

### 3.2.连接接自定义MCP服务

注意：由于我们自定义的MCP是本地py文件，所以启动的Command直接就是python，而不是npx或uvx


In [17]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 连接自定义的mcp服务
client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["./math_mcp_server.py"]
        }
    }
)

tools = await client.get_tools()
for tool in tools:
    print(tool.name)
    print(tool.description)
    print('---------------------------')

add
add two numbers
---------------------------
multiply
Multiply two numbers
---------------------------
square_root
Calculate the square root of a number
---------------------------


In [18]:
from langchain.agents import create_agent

# 创建智能体
agent = create_agent(
    model="deepseek-flash",
    tools=tools,
    system_prompt="你是一个智能助手，你必须使用工具来回答数学问题"
)

In [20]:
from langchain.messages import HumanMessage

# 调用agent
response = await agent.ainvoke(
    {"messages": [HumanMessage("444和1234的平方根之和是多少")]}
)

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

444和1234的平方根之和是多少
================================== Ai Message ==================================
Tool Calls:
  square_root (call_00_Ggpamu76n5Vd4N6tkohn8114)
 Call ID: call_00_Ggpamu76n5Vd4N6tkohn8114
  Args:
    x: 444
  square_root (call_01_CJBRtuA9KXrmzOnqLLe76559)
 Call ID: call_01_CJBRtuA9KXrmzOnqLLe76559
  Args:
    x: 1234
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '21.071307505705477', 'id': 'lc_e00b320d-778a-465a-8530-b9abbe71107a'}]
================================= Tool Message =================================
Name: square_root

[{'type': 'text', 'text': '35.12833614050059', 'id': 'lc_c9a4b5e0-8d2a-4165-824f-92df7ac4fcaf'}]
================================== Ai Message ==================================
Tool Calls:
  add (call_00_F7j7JGN6aGMtPINKRZ733136)
 Call ID: call_00_F7j7JGN6aGMtPINKRZ733136
  Args:
    